# **Duelo de Modelos: XBoost v.s. Random Forest**

Neste projeto, o desempenho dos modelos Xboost e Random Forest serão comparados. A base a ser analisada contém os dados dos passageiros do Titanic na sua última viagem, e a informação de quais sobreviveram ao acidente e quais não. Os dados estão disponíveis em https://www.kaggle.com/c/titanic .

## Tratamento dos dados de treinamento

Foram excluídos as categorias não relevantes da base, como número do ticket e nome do passageiro. Além disso, foram avaliados os tipos de dados, erros de digitação e coerência das notações.

A categoria 'cabine' foi alterada para binário. As células vazias, referentes aos passageiros que não estavam na primeira classe,  cabine, foi transformada em 0, enquanto as de mais células, que continham a informção da cabine, ou seja, eram passageiros de primeira classe, foi alterada para 1.

Além disso, as variáveis 'fare' e 'sex' foram corrigidas e transformadas em binárias, enquanto a categoria 'embarked' foi deocmposta em outras três categorias binárias, uma para cada tipo de embarja. Por úlitmo, os valores nulos da variável 'idade' foi corrigida para a média das idades.

In [41]:
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np

In [42]:
xy_train = pd.read_csv('train.csv')
x_test = pd.read_csv('test.csv')
PassengerId = x_test['PassengerId']
xy_train.drop(['Ticket', 'Name', 'PassengerId'], axis=1, inplace=True)
x_test.drop(['Ticket', 'Name', 'PassengerId'], axis=1, inplace=True)
print(xy_train.columns,"\n", x_test.columns)

Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin',
       'Embarked'],
      dtype='object') 
 Index(['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin', 'Embarked'], dtype='object')


In [43]:
xy_train.head(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
0,0,3,male,22.0,1,0,7.2500,NaN,S
1,1,1,female,38.0,1,0,71.2833,C85,C
2,1,3,female,26.0,0,0,7.9250,NaN,S
3,1,1,female,35.0,1,0,53.1000,C123,S
4,0,3,male,35.0,0,0,8.0500,NaN,S


In [44]:
xy_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    object 
 3   Age       714 non-null    float64
 4   SibSp     891 non-null    int64  
 5   Parch     891 non-null    int64  
 6   Fare      891 non-null    float64
 7   Cabin     204 non-null    object 
 8   Embarked  889 non-null    object 
dtypes: float64(2), int64(4), object(3)
memory usage: 62.8+ KB


In [45]:

xy_train['Cabin'] = xy_train['Cabin'].notna().astype(int)
xy_train.fillna({'Cabin':0}, inplace=True)

x_test['Cabin'] = x_test['Cabin'].notna().astype(int)
x_test.fillna({'Cabin':0}, inplace=True)

print(xy_train['Cabin'].unique(),"\n",x_test['Cabin'].unique())


[0 1] 
 [0 1]


In [46]:
print(xy_train.dtypes)

Survived      int64
Pclass        int64
Sex          object
Age         float64
SibSp         int64
Parch         int64
Fare        float64
Cabin         int64
Embarked     object
dtype: object


In [47]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
xy_train['Sex'] = label_encoder.fit_transform(xy_train['Sex'])
x_test['Sex'] = label_encoder.fit_transform(x_test['Sex'])
# Oneenconder
xy_train = pd.get_dummies(xy_train, columns=['Embarked'], prefix='Embarked')
x_test = pd.get_dummies(x_test, columns=['Embarked'], prefix='Embarked')
print(xy_train.dtypes)

Survived        int64
Pclass          int64
Sex             int64
Age           float64
SibSp           int64
Parch           int64
Fare          float64
Cabin           int64
Embarked_C       bool
Embarked_Q       bool
Embarked_S       bool
dtype: object


In [48]:
xy_train['Age'] = xy_train['Age'].fillna(xy_train['Age'].mean())
x_test['Age'] = x_test['Age'].fillna(x_test['Age'].mean())


## Busca por outliers e balancemaento

A seguir, seguem os histogramas e box plots, quando adequado, das categorias analisadas. A categoria alvo, 'Survived', está razoavelmente equilibrada, especialmente ao ser considerado o contexto de origem dos dados. De maneira geral, as variaveis preditórias não apresentam comportamento estranho nem desbalanceado.  

In [49]:
for campo in xy_train.columns:
    if xy_train[campo].dtype in  ['int64', 'int32', 'float64', 'float32']:
        fig = make_subplots(rows=1, cols=2, subplot_titles=[f'Histograma de {campo}', f'Box Plot de {campo}'])
        fig.add_trace(
            px.histogram(xy_train, x=campo, histnorm="percent", nbins=60).data[0],
            row=1, col=1
        )

        fig.add_trace(
            px.box(xy_train, y=campo).data[0],
            row=1, col=2
            )
        fig.update_layout(title_text=f'{campo}', showlegend=False)
        fig.show()
    else:
        fig = px.histogram(xy_train, x=campo, histnorm="percent", nbins=60)
        fig.update_layout(title_text=f'{campo}', showlegend=False)
        fig.show()


## Normalização dos dados
A seguir, a normalização da base com a técnica SMOTE.

In [50]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin',
       'Embarked_C', 'Embarked_Q', 'Embarked_S']



In [51]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
x, y = smote.fit_resample(xy_train[features], xy_train['Survived'])
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)
x_test_scaled = scaler.transform(x_test)

## Aperfeiçoamento dos modelos

Para o aperfeiçoamento dos modelos a serem criados, os hiperparâmetros foram criados com o GridSearchCV para o XBoost e RandomizedSearch para o Random Forest. Adicionalmmente, foi feita a aplicação do cross validation a fim de melhorar a robusteza deles.


In [55]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, accuracy_score
from scipy.stats import randint

## XBoost

O modelo XBoost foi criado com os parâmetros indicados no código.

O modelo apresentou acurácia de 0.92, com f1-score de 0.92.

In [56]:
import xgboost as xgb

In [57]:
param_grid_xgboost = {
    'max_depth': [3, 5, 7],              
    'n_estimators': [50, 100, 200],     
    'learning_rate': [0.01, 0.1, 0.2],   
    'subsample': [0.8, 1.0],           
    'colsample_bytree': [0.8, 1.0]       
}

xgboost = xgb.XGBClassifier()

grid_search_xgboost = GridSearchCV(
    estimator = xgboost,   
    param_grid=param_grid_xgboost,     
    scoring='accuracy',         
    cv=5,                       
    n_jobs=-1                  
)

grid_search_xgboost.fit(x_scaled, y)

print("Melhores Parâmetros:", grid_search_xgboost.best_params_)
print("Melhor Acurácia:", grid_search_xgboost.best_score_)

best_model_xgboost = grid_search_xgboost.best_estimator_
y_pred_xgboost = best_model_xgboost.predict(x_test_scaled)

y_pred_train = best_model_xgboost.predict(x_scaled)
accuracy = accuracy_score(y, y_pred_train)
print(f"Acurácia: {accuracy:.2f}")

report = classification_report(y, y_pred_train)
print("Relatório de Classificação:")
print(report)

df_xb = pd.DataFrame()
df_xb['Survived'] = y_pred_xgboost
df_xb['PassengerId'] = PassengerId
df_xb.to_csv('y_pred_xboost.csv', index=False)


Melhores Parâmetros: {'colsample_bytree': 1.0, 'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 50, 'subsample': 0.8}
Melhor Acurácia: 0.860763802407638
Acurácia: 0.92
Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.90      0.95      0.92       549
           1       0.94      0.90      0.92       549

    accuracy                           0.92      1098
   macro avg       0.92      0.92      0.92      1098
weighted avg       0.92      0.92      0.92      1098



## Random Forest

O modelo Random Forest foi criado com os parâmetros indicados no código.

O modelo apresentou acurácia de 0.93, com f1-score de 0.93.

In [58]:
from sklearn.ensemble import RandomForestClassifier

In [60]:
# Hyperparametros
params_grid_rf = {
    'n_estimators': randint(50, 200),
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': randint(2, 10)
}

randomforest = RandomForestClassifier(random_state=42)

rf_search = RandomizedSearchCV(
    estimator= randomforest,
    param_distributions= params_grid_rf,
    n_iter=100,  
    random_state=42,
    cv = 5,
    n_jobs=-1  )

rf_search.fit(x_scaled, y)

best_params_rf = rf_search.best_params_
print("Melhores Parâmetros Encontrados:", best_params_rf)

y_pred_train = rf_search.predict(x_scaled)
accuracy = accuracy_score(y, y_pred_train)
print(f"Acurácia: {accuracy:.2f}")

report = classification_report(y, y_pred_train)
print("Relatório de Classificação:")
print(report)

y_pred_rf = rf_search.predict(x_test_scaled)

df_rf = pd.DataFrame()
df_rf['Survived'] = y_pred_rf
df_rf['PassengerId'] = PassengerId
df_rf.to_csv('y_pred_rf.csv', index=False)


Melhores Parâmetros Encontrados: {'max_depth': 10, 'min_samples_split': 4, 'n_estimators': 196}
Acurácia: 0.93
Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.91      0.96      0.93       549
           1       0.96      0.90      0.93       549

    accuracy                           0.93      1098
   macro avg       0.93      0.93      0.93      1098
weighted avg       0.93      0.93      0.93      1098



# Dicas de Melhoria

- O target (Survived) do Titanic é desbalanceado. Para o XGBoost, utilize o parâmetro scale_pos_weight e, para o Random Forest, utilize o parâmetro class_weight='balanced' para que o modelo dê maior atenção à classe minoritária, potencialmente melhorando o Recall e o score final.
- O seu trabalho menciona o duelo, mas o output foca apenas no Random Forest. Uma dica é adicionar a comparação explícita das métricas (ex: AUC, F1-Score) dos dois modelos (XGBoost e Random Forest) no conjunto de teste, garantindo que o modelo escolhido para a submissão (Random Forest) seja, de fato, o de melhor desempenho.